<div style="text-align: center; padding-top: 30px; padding-bottom: 10px;">
<h1>Levels model</h1>
<p>Climate response surface for the levels specification, where the dependent
variable is log GDP per capita rather than the growth rate.</p>
</div>

In [1]:
import sys, yaml
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
from utils import create_pred_input, add_histogram
from hdf5_min import read_weights

############################## User Input ############





# 
# ##################
run_dir  = ROOT / "runs/estimation/2026-09-28_20-15-05_global_IC_country_trends_levels"
n_models = 6           # how many architectures to overlay
########################################################################

cfg = yaml.safe_load((run_dir / ".hydra/config.yaml").read_text(encoding="utf-8"))["instance"]
m = cfg["model"]
print(run_dir.name)
for k in ("target_mode", "dynamic_model", "country_trends", "quadratic_trends",
          "output_initializer"):
    print(f"  {k:20} {m.get(k)}")
print(f"  {'no_inits':20} {cfg['optim'].get('no_inits')}")
print(f"  {'architectures':20} {len(m.get('nodes_list', []))}")
assert str(m.get("target_mode")).lower() == "levels", "this notebook expects a levels run"

2026-09-28_20-15-05_global_IC_country_trends_levels
  target_mode          levels
  dynamic_model        False
  country_trends       True
  quadratic_trends     True
  output_initializer   zeros
  no_inits             40
  architectures        54


## 1. Data preparation

In [2]:
data = pd.read_excel(ROOT / "data/MainData.xlsx")
if m.get("data_end"):
    data = data[data.Year <= m["data_end"]]

mean_temp   = float(np.nanmean(data["TempPopWeight"]))
std_temp    = float(np.nanstd(data["TempPopWeight"]))
mean_precip = float(np.nanmean(data["PrecipPopWeight"]))
std_precip  = float(np.nanstd(data["PrecipPopWeight"]))

# Same prediction grid as the other notebooks: precipitation capped at 3 m.
pred_input, T, P = create_pred_input(mc=False, mean_T=mean_temp, std_T=std_temp,
                                     mean_P=mean_precip, std_P=std_precip,
                                     precip_capped=True)
temp_vals, precip_vals = T[0, :], P[:, 0]

# Supported range: only 3 of 10,324 country-years exceed 30 C, so anything beyond
# the 5th-95th percentile band is functional form rather than evidence.
Tobs = data["TempPopWeight"].to_numpy(float)
t_lo, t_hi = np.nanpercentile(Tobs[np.isfinite(Tobs)], [5, 95])
precip_med = float(np.nanmedian(data["PrecipPopWeight"]))

print(f"grid {T.shape}, temperature {temp_vals[0]:.1f}-{temp_vals[-1]:.1f} C, "
      f"precipitation {precip_vals[0]:.0f}-{precip_vals[-1]:.0f} mm")
print(f"temperature {mean_temp:.2f} +/- {std_temp:.2f} C, supported {t_lo:.1f}-{t_hi:.1f} C")
print(f"precipitation median {precip_med:.0f} mm")

grid (90, 90), temperature 0.0-30.0 C, precipitation 12-3000 mm
temperature 20.31 +/- 7.27 C, supported 6.9-28.1 C
precipitation median 1118 mm


## 2. Model overview

In [3]:
results = dict(np.load(run_dir / "results.npy", allow_pickle=True).item())
results = {k: v for k, v in results.items() if v is not None and np.isfinite(v[0])}

# results.npy is {node: [BIC, AIC]}. Rank on BIC, index 0.
top_models = sorted(results, key=lambda node: results[node][0])[:n_models]

best = results[top_models[0]][0]
print(f"{'arch':>12}{'BIC':>13}{'AIC':>13}{'dBIC':>9}")
for node in top_models:
    b, a = results[node][0], results[node][1]
    print(f"{str(node):>12}{b:13.2f}{a:13.2f}{b - best:9.2f}")

        arch          BIC          AIC     dBIC
        (2,)    -37843.25    -42594.15     0.00
      (2, 2)    -37804.82    -42599.18    38.42
      (4, 2)    -37787.37    -42654.14    55.88
        (4,)    -37785.13    -42593.97    58.11
   (4, 2, 2)    -37779.13    -42689.36    64.12
   (2, 2, 2)    -37770.99    -42608.80    72.26


## 3. Prediction on the temperature-precipitation grid

The network output is the climate component of log GDP per capita, with country
and time fixed effects and country trends partialled out. The fixed effects absorb
the overall level, so the surface is centred at the sample-mean climate: the
vertical axis reads as log GDP per capita relative to a country experiencing
average temperature and precipitation.

In [4]:
def swish(z):
    return z / (1.0 + np.exp(-np.clip(z, -60.0, 60.0)))


def forward(w, X):
    """Chained Dense stack: hidden layers carry a bias, the output layer does not."""
    idx = sorted({int(k.split("/")[1].split("_")[1]) if "_" in k.split("/")[1] else 0
                  for k in w if k.startswith("layers")})
    h = X
    for i in idx:
        b = f"layers/dense{'' if i == 0 else f'_{i}'}/vars"
        h = h @ np.asarray(w[f"{b}/0"], float)
        if f"{b}/1" in w:
            h = swish(h + np.asarray(w[f"{b}/1"], float))
    return h[..., 0]


def surface(node, weights_path=None):
    """Centred levels surface on the (T, P) grid, in log points."""
    p = weights_path or (run_dir / "parameters" / f"{node}.weights.h5")
    w = read_weights(str(p))
    Z = forward(w, pred_input).reshape(T.shape)
    origin = np.zeros((1,) * (pred_input.ndim - 1) + (pred_input.shape[-1],), dtype=float)
    centre = float(np.ravel(forward(w, origin))[0])              # standardised origin
    return Z - centre


model_surfaces = {node: surface(node) for node in top_models
                  if (run_dir / "parameters" / f"{node}.weights.h5").exists()}
mean_surface = np.mean(list(model_surfaces.values()), axis=0)

for node, Z in model_surfaces.items():
    print(f"  {str(node):>12}  range {Z.min():+.3f} to {Z.max():+.3f} log points")
print(f"  {'mean':>12}  range {mean_surface.min():+.3f} to {mean_surface.max():+.3f}")

          (2,)  range -0.111 to +0.084 log points
        (2, 2)  range -0.107 to +0.267 log points
        (4, 2)  range -0.200 to +0.175 log points
          (4,)  range -0.108 to +0.049 log points
     (4, 2, 2)  range -0.303 to +0.103 log points
     (2, 2, 2)  range -0.178 to +0.114 log points
          mean  range -0.128 to +0.087


## 4. Data visualization

In [5]:
############################## User Input ##############################
show    = "best"          # "best", "mean", or "all"
z_range = [-0.3, 0.3]     # fixed, as in global_model.ipynb and paper_surface.py,
                          # so the viewing geometry matches. None to autoscale.
########################################################################

if show == "mean":
    surfaces = [("mean", mean_surface)]
elif show == "all":
    surfaces = list(model_surfaces.items())
else:
    surfaces = [(top_models[0], model_surfaces[top_models[0]])]

fig = go.Figure()
for name, Z in surfaces:
    fig.add_trace(go.Surface(
        x=T, y=P / 1000, z=Z,          # precipitation in metres, as in the paper
        colorscale="Cividis",
        opacity=0.85 if len(surfaces) == 1 else 0.45,
        showscale=False, name=str(name)))

# Observed climate density on the floor, as in the other notebooks.
fig = add_histogram(fig, data, legend=False, precip_capped=True, z_floor=-0.3)

zaxis = dict(title=dict(text="ln(GDP)"))
if z_range is not None:
    zaxis["range"] = list(z_range)

fig.update_layout(
    autosize=True,
    margin=dict(l=0, r=0, b=0, t=0),
    scene=dict(
        xaxis_title="Temperature (°C)",
        yaxis_title="Precipitation (m)",
        zaxis=zaxis,
        camera=dict(eye=dict(x=1.738, y=-1.780, z=0.589)),
    ),
    legend=dict(bgcolor="rgba(255,255,255,0.7)", bordercolor="black", borderwidth=1),
    font=dict(size=10),
)
fig.show()

## 6. Export

The HTML goes to `results/images/` for interactive inspection; the PDF goes
straight into `paper/Figures/Levels/`, which is the path Appendix
`sec:Appendix_Levels` includes. Writing the PDF anywhere else means the paper
silently keeps compiling against an older file.

In [ ]:
# Export. The PDF is cropped on the way out, so no Acrobat "Set Page Box" step
# is needed.
#
# The crop is taken from a reference figure rather than from margin numbers:
# Plotly writes every one of these surfaces on the same 1350 x 1050 pt page with
# the same camera, so copying the reference's CropBox reproduces its framing
# exactly and keeps the whole figure set visually consistent. If the reference
# is missing, it falls back to margins.
from pypdf import PdfReader, PdfWriter

CROP_REFERENCE = ROOT / "paper/Figures/Global/global_top10surfaces_CT_fit.pdf"

# Nudges applied to the reference box, in points. Negative moves an edge
# outwards (less crop), positive inwards (more crop). Kept identical to
# global_model.ipynb so the levels surface is framed like the growth surface.
# Edit THESE to adjust the framing. MARGINS_MM below is only the fallback used
# when the reference file is missing, and has no effect while it exists.
CROP_ADJUST_PT = {"left": -40.0, "bottom": 0.0, "right": 30.0, "top": 30.0}
MARGINS_MM = {"top": 70.0, "bottom": 28.6, "left": 75.0, "right": 112}
MM_TO_PT = 72.0 / 25.4


def reference_box(path, adjust=None):
    """CropBox of the reference, nudged, in the original page coordinate system."""
    a = adjust or {}
    box = PdfReader(str(path)).pages[0].cropbox
    return (float(box.left) + a.get("left", 0.0),
            float(box.bottom) + a.get("bottom", 0.0),
            float(box.right) + a.get("right", 0.0),
            float(box.top) - a.get("top", 0.0))


def crop_pdf(path, reference=CROP_REFERENCE, margins_mm=MARGINS_MM):
    """Set CropBox and MediaBox in place, matching `reference` when available."""
    box = reference_box(reference, CROP_ADJUST_PT) if reference and Path(reference).exists() else None

    reader, writer = PdfReader(str(path)), PdfWriter()
    for page in reader.pages:
        page_box = page.mediabox
        if box is not None:
            left, bottom, right, top = box
        else:
            left = float(page_box.left) + margins_mm["left"] * MM_TO_PT
            bottom = float(page_box.bottom) + margins_mm["bottom"] * MM_TO_PT
            right = float(page_box.right) - margins_mm["right"] * MM_TO_PT
            top = float(page_box.top) - margins_mm["top"] * MM_TO_PT
        if right <= left or top <= bottom:
            raise ValueError(f"crop box is empty for {path}")
        page.cropbox.lower_left = (left, bottom)
        page.cropbox.upper_right = (right, top)
        page.mediabox.lower_left = (left, bottom)
        page.mediabox.upper_right = (right, top)
        writer.add_page(page)
    with open(path, "wb") as fh:
        writer.write(fh)
    return (left, bottom, right, top), box is not None


html_dir = ROOT / "results/images"
html_dir.mkdir(parents=True, exist_ok=True)
html = html_dir / f"levels_surface_{run_dir.name}.html"
fig.write_html(html, include_plotlyjs="cdn")

pdf_dir = ROOT / "paper/Figures/Levels"
pdf_dir.mkdir(parents=True, exist_ok=True)
pdf = pdf_dir / "levels_surface.pdf"

# width/height/scale here produce a 1350 x 1050 pt page, which is the page the
# reference CropBox was measured on. Changing them silently reframes the figure,
# so the page size is checked rather than assumed.
fig.write_image(pdf, width=900, height=700, scale=2)

_pg = PdfReader(str(pdf)).pages[0].mediabox
_w, _h = float(_pg.width), float(_pg.height)
if abs(_w - 1350) > 1 or abs(_h - 1050) > 1:
    print(f"   WARNING: page is {_w:.0f} x {_h:.0f} pt, not 1350 x 1050; "
          f"the reference crop box will not frame this figure correctly")

(l, b, r, t), matched = crop_pdf(pdf)

print(f"-> {html}")
print(f"-> {pdf}")
print(f"   crop [{l:.3f} {b:.4f} {r:.3f} {t:.3f}]  "
      f"({r - l:.1f} x {t - b:.1f} pt)  "
      + (f"matched {CROP_REFERENCE.name}" if matched else "margin fallback"))
